<a href="https://colab.research.google.com/github/RiccoFlores/100-Days-Of-ML-Code/blob/master/NB5_Tidy_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

## 1. Pew Data

Income Distribution Within U.S. Religious Groups ([Link](
https://www.pewforum.org/2009/01/30/income-distribution-within-us-religious-groups/ ))

In [ ]:
pew_data = pd.read_csv('pew.txt', sep = "\t")

In [ ]:
pew_data.head()

,religion,<$10k,$10-20k,$20-30k,$30-40k,$40-50k,$50-75k,$75-100k,$100-150k,>150k,Don't know/refused
0,Agnostic,27,34,60,81,76,137,122,109,84,96
1,Atheist,12,27,37,52,35,70,73,59,74,76
2,Buddhist,27,21,30,34,33,58,62,39,53,54
3,Catholic,418,617,732,670,638,1116,949,792,633,1489
4,Don’t know/refused,15,14,15,11,10,35,21,17,18,116


In [ ]:
# Melting income into a unique column
pew_tidy = pd.melt(frame = pew_data,
                   id_vars = "religion",
                   var_name = "income",
                   value_name = "freq")

In [ ]:
pew_tidy.head()

,religion,income,freq
0,Agnostic,<$10k,27
1,Atheist,<$10k,12
2,Buddhist,<$10k,27
3,Catholic,<$10k,418
4,Don’t know/refused,<$10k,15


## 2. Weather

In [ ]:
weather_data = pd.read_csv('weather.txt', sep = "\t")

In [ ]:
weather_data.head()

,id,year,month,element,d1,d2,d3,d4,d5,d6,...,d22,d23,d24,d25,d26,d27,d28,d29,d30,d31
0,MX000017004,2010,1,TMAX,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,278.0,NaN
1,MX000017004,2010,1,TMIN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,145.0,NaN
2,MX000017004,2010,2,TMAX,NaN,273.0,241.0,NaN,NaN,NaN,...,NaN,299.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MX000017004,2010,2,TMIN,NaN,144.0,144.0,NaN,NaN,NaN,...,NaN,107.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MX000017004,2010,3,TMAX,NaN,NaN,NaN,NaN,321.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Melting day (d1, d2, ..., d31) columns into a unique column
weather_tidy = pd.melt(frame = weather_data,
                       id_vars = ["id", "year", "month", "element"],
                       var_name = "day",
                       value_name = "value")

In [ ]:
weather_tidy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 682 entries, 0 to 681
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       682 non-null    object 
 1   year     682 non-null    int64  
 2   month    682 non-null    int64  
 3   element  682 non-null    object 
 4   day      682 non-null    object 
 5   value    66 non-null     float64
dtypes: float64(1), int64(2), object(3)
memory usage: 32.1+ KB


In [ ]:
# Data Cleaning Steps
# 1. Delete NaN values
weather_tidy.dropna(inplace = True)

# 2. Clean column "day" (d1 -> 1)
weather_tidy["day"] = weather_tidy["day"].str.replace("d", "").astype(int)

# 3. Create a "date" column
# (2010, 1, 1) -> "2010-1-1"
weather_tidy["date"] = weather_tidy["year"].astype(str) + '-' + weather_tidy["month"].astype(str) + '-' + weather_tidy["day"].astype(str)

# 4. Parse "date" column
weather_tidy["date_formated"] = pd.to_datetime(weather_tidy["date"])

# 5. Drop columns (year, month, day)
weather_tidy.drop(columns = ["year", "month", "day"], inplace = True)

# 6. Reset index
weather_tidy.reset_index(drop = True, inplace = True)

In [ ]:
weather_tidy.head()

,element,value,date_formated
0,TMAX,299.0,2010-12-01
1,TMIN,138.0,2010-12-01
2,TMAX,273.0,2010-02-02
3,TMIN,144.0,2010-02-02
4,TMAX,313.0,2010-11-02


In [ ]:
weather_tidy.drop(columns=["id", "date"],inplace = True)

In [ ]:
weather_tidy = weather_tidy.pivot_table(index = ["date_formated"],
                                        columns = "element",
                                        values = "value").reset_index()

weather_tidy.head()

KeyError: 'value'

In [ ]:
weather_tidy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date_formated  33 non-null     datetime64[ns]
 1   TMAX           33 non-null     float64       
 2   TMIN           33 non-null     float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 924.0 bytes


In [ ]:
weather_tidy.rename(index={'element':'index'},inplace=True)
weather_tidy.head()

element,date_formated,TMAX,TMIN
0,2010-01-30,278.0,145.0
1,2010-02-02,273.0,144.0
2,2010-02-03,241.0,144.0
3,2010-02-11,297.0,134.0
4,2010-02-23,299.0,107.0


## 3. Baby Names

In [ ]:
babynames_2014_data = pd.read_csv('2014-baby-names-illinois.csv')
babynames_2015_data = pd.read_csv('2015-baby-names-illinois.csv')

In [ ]:
babynames_2014_data.head()

,rank,name,frequency,sex,year
0,1,Noah,837,Male,2014
1,2,Alexander,747,Male,2014
2,3,William,687,Male,2014
3,4,Michael,680,Male,2014
4,5,Liam,670,Male,2014


In [ ]:
babynames_2015_data.head()

,rank,name,frequency,sex,year
0,1,Noah,863,Male,2015
1,2,Liam,709,Male,2015
2,3,Alexander,703,Male,2015
3,4,Jacob,650,Male,2015
4,5,William,618,Male,2015


In [ ]:
babynames_2014_data["year"] = 2014
babynames_2015_data["year"] = 2015

In [ ]:
babynames = pd.concat([babynames_2014_data, babynames_2015_data])

In [ ]:
babynames.reset_index(inplace=True)

In [ ]:
babynames.tail()

,index,rank,name,frequency,sex,year
196,95,96,Giovanni,168,Male,2015
197,96,97,Hudson,167,Male,2015
198,97,98,Camden,165,Male,2015
199,98,99,Max,164,Male,2015
200,99,100,Maxwell,155,Male,2015


In [ ]:
babynames_2014_data.shape, babynames_2015_data.shape, babynames.shape

((101, 5), (100, 5), (201, 6))

##4. Billboard

In [ ]:
billboard_df = pd.read_csv('billboard.csv', encoding='latin1')

In [ ]:
billboard_df.head()

,year,artist.inverted,track,time,genre,date.entered,date.peaked,x1st.week,x2nd.week,x3rd.week,...,x67th.week,x68th.week,x69th.week,x70th.week,x71st.week,x72nd.week,x73rd.week,x74th.week,x75th.week,x76th.week
0,2000,Destiny's Child,Independent Women Part I,3:38,Rock,2000-09-23,2000-11-18,78,63.0,49.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000,Santana,"Maria, Maria",4:18,Rock,2000-02-12,2000-04-08,15,8.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000,Savage Garden,I Knew I Loved You,4:07,Rock,1999-10-23,2000-01-29,71,48.0,43.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000,Madonna,Music,3:45,Rock,2000-08-12,2000-09-16,41,23.0,18.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000,"Aguilera, Christina",Come On Over Baby (All I Want Is You),3:38,Rock,2000-08-05,2000-10-14,57,47.0,45.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
billboard_melt = pd.melt(frame = billboard_df,
                         id_vars = ["year", "artist.inverted", "track", "time", 'genre', 'date.entered', 'date.peaked'],
                         var_name = "week",
                         value_name = "rank")

In [ ]:
billboard_melt.head()

,year,artist.inverted,track,time,genre,date.entered,date.peaked,week,rank
0,2000,Destiny's Child,Independent Women Part I,3:38,Rock,2000-09-23,2000-11-18,1st.week,78.0
1,2000,Santana,"Maria, Maria",4:18,Rock,2000-02-12,2000-04-08,1st.week,15.0
2,2000,Savage Garden,I Knew I Loved You,4:07,Rock,1999-10-23,2000-01-29,1st.week,71.0
3,2000,Madonna,Music,3:45,Rock,2000-08-12,2000-09-16,1st.week,41.0
4,2000,"Aguilera, Christina",Come On Over Baby (All I Want Is You),3:38,Rock,2000-08-05,2000-10-14,1st.week,57.0


In [ ]:
#Manejo de la columna week
# x1st.week, x10th.week
billboard_melt['week'] = billboard_melt['week'].str.replace('x', '')

In [ ]:
billboard_melt['week'][1000][:-7]

'4'

In [ ]:
#list comprenhension
billboard_melt["week_clean"] = [w[:-7] for w in billboard_melt["week"]]

In [ ]:
billboard_melt.head()

,year,artist.inverted,track,time,genre,date.entered,date.peaked,week,rank,week_clean
0,2000,Destiny's Child,Independent Women Part I,3:38,Rock,2000-09-23,2000-11-18,1st.week,78.0,1
1,2000,Santana,"Maria, Maria",4:18,Rock,2000-02-12,2000-04-08,1st.week,15.0,1
2,2000,Savage Garden,I Knew I Loved You,4:07,Rock,1999-10-23,2000-01-29,1st.week,71.0,1
3,2000,Madonna,Music,3:45,Rock,2000-08-12,2000-09-16,1st.week,41.0,1
4,2000,"Aguilera, Christina",Come On Over Baby (All I Want Is You),3:38,Rock,2000-08-05,2000-10-14,1st.week,57.0,1


## 5. Tubercolosis

In [ ]:
tb_df = pd.read_csv('tb.csv')

In [ ]:
tb_df.head()

,iso2,year,new_sp,new_sp_m04,new_sp_m514,new_sp_m014,new_sp_m1524,new_sp_m2534,new_sp_m3544,new_sp_m4554,...,new_sp_f04,new_sp_f514,new_sp_f014,new_sp_f1524,new_sp_f2534,new_sp_f3544,new_sp_f4554,new_sp_f5564,new_sp_f65,new_sp_fu
0,AD,1989,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AD,1990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AD,1991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AD,1992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AD,1993,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
tb_melt = pd.melt(frame = tb_df,
                  id_vars = ["iso2", "year"],
                  var_name = "sex_age",
                  value_name = "cases")

In [ ]:
tb_melt.sex_age.unique()

array(['new_sp', 'new_sp_m04', 'new_sp_m514', 'new_sp_m014',
       'new_sp_m1524', 'new_sp_m2534', 'new_sp_m3544', 'new_sp_m4554',
       'new_sp_m5564', 'new_sp_m65', 'new_sp_mu', 'new_sp_f04',
       'new_sp_f514', 'new_sp_f014', 'new_sp_f1524', 'new_sp_f2534',
       'new_sp_f3544', 'new_sp_f4554', 'new_sp_f5564', 'new_sp_f65',
       'new_sp_fu'], dtype=object)

In [ ]:
rename_rules = {'new_sp':'total',
                'new_sp_m04':'m0-4',
                'new_sp_m514':'m5-14',
                'new_sp_m014':'m0-14',
                'new_sp_m1524':'m15-24',
                'new_sp_m2534':'m25-34',
                'new_sp_m3544':'m35-44',
                'new_sp_m4554':'m45-54',
                'new_sp_m5564':'m55-64',
                'new_sp_m65':'m65',
                'new_sp_mu':'mu',
                'new_sp_f04':'f0-4',
                'new_sp_f514':'f5-14',
                'new_sp_f014':'f0-14',
                'new_sp_f1524':'f15-24',
                'new_sp_f2534':'f25-34',
                'new_sp_f3544':'f35-44',
                'new_sp_f4554':'f45-54',
                'new_sp_f5564':'f55-64',
                'new_sp_f65':'f65',
                'new_sp_fu':'fu'}

tb_melt['sex_age'].replace(rename_rules, inplace=True)

In [ ]:
# Source - https://stackoverflow.com/a/37683738
# Posted by Jon Clements, modified by community. See post 'Timeline' for change history
# Retrieved 2026-02-18, License - CC BY-SA 4.0
#df.A.str.extract('(\d+)')

tb_melt['age'] = tb_melt['sex_age'].str.extract('(\d+)')

<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-131848144.py:6: SyntaxWarning: invalid escape sequence '\d'
  tb_melt['age'] = tb_melt['sex_age'].str.extract('(\d+)')


In [ ]:
def asignar_genero(valor):
  if valor.startswith('new_sp_m'):
    return 'Male'
  elif valor.startswith('new_sp_f'):
    return 'Female'
  else:
    return 'Unknown'

In [ ]:
tb_melt["genero"] = tb_melt["sex_age"].apply(asignar_genero)

In [ ]:
tb_melt.age.unique()

array([nan, '0', '5', '15', '25', '35', '45', '55', '65'], dtype=object)